In [1]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
from datetime import datetime
import numpy as np

# Set tracking URI to match your running server instance
mlflow.set_tracking_uri('http://localhost:5000')

# Create the backend MLflow tracking client
client = MlflowClient()

# Set and verify the experiment context
experiment = client.get_experiment_by_name('predictive-maintenance')
print(f'Experiment Name: {experiment.name}')
print(f'Experiment ID:   {experiment.experiment_id}')
print("-" * 60)

# Get all runs from the experiment, sorted by highest ROC AUC
runs = client.search_runs(
    experiment_ids=experiment.experiment_id,
    order_by=['metrics.roc_auc DESC']
)

# Display runs table neatly
run_data = []
for run in runs:
    run_data.append({
        'run_id': run.info.run_id[:8],
        'name': run.info.run_name,
        'model_type': run.data.params.get('model_type', 'Unknown'),
        'roc_auc': run.data.metrics.get('roc_auc', 0),
        'f1_score': run.data.metrics.get('f1_score', 0),
        'accuracy': run.data.metrics.get('accuracy', 0)
    })

df = pd.DataFrame(run_data)
print('All Runs (sorted by ROC AUC):')
print(df.to_string(index=False))

Experiment Name: predictive-maintenance
Experiment ID:   2
------------------------------------------------------------
All Runs (sorted by ROC AUC):
  run_id                name         model_type  roc_auc  f1_score  accuracy
6f08cd37       random_forest       RandomForest 0.975080  0.597561     0.967
ea7990b7             xgboost            XGBoost 0.970920  0.592593     0.967
3a0e409c logistic_regression LogisticRegression 0.923425  0.254902     0.962


In [2]:
# Target the best run (index 0 on our sorted leaderboard list)
best_run = runs[0]
best_run_id = best_run.info.run_id

print(f'Best model selected: {best_run.info.run_name}')
print(f'Highest ROC AUC:     {best_run.data.metrics["roc_auc"]:.4f}')
print("-" * 60)

# Register the model formally in the central hub
model_name = 'PredictiveMaintenance'
model_uri = f'runs:/{best_run_id}/model'

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f'\n[SUCCESS] Registered "{model_name}" as Version {registered_model.version}!')
print("👉 Go check the 'Models' tab in your browser at http://localhost:5000 to see it live!")

Best model selected: random_forest
Highest ROC AUC:     0.9751
------------------------------------------------------------


Successfully registered model 'PredictiveMaintenance'.
2026/06/07 10:19:22 WARNING mlflow.tracking._model_registry.fluent: Run with id 6f08cd37c3b4475fae0a0390c93aeef6 has no artifacts at artifact path 'model', registering model based on models:/m-ce1d360fb8004a10a20df6d879b6019d instead
2026/06/07 10:19:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PredictiveMaintenance, version 1



[SUCCESS] Registered "PredictiveMaintenance" as Version 1!
👉 Go check the 'Models' tab in your browser at http://localhost:5000 to see it live!


Created version '1' of model 'PredictiveMaintenance'.


In [3]:
version = registered_model.version
roc_auc = best_run.data.metrics['roc_auc']
f1 = best_run.data.metrics['f1_score']

# Generate crisp documentation markdown
description = f'''### Predictive Maintenance Engine
**Model Type:** XGBoost Classifier
**Training Size:** 10,000 synthetic machinery rows.

#### Baseline Performance:
* **ROC AUC:** {roc_auc:.4f}
* **F1 Score:** {f1:.4f}

*Features utilized include temperature, vibration, pressure, rpm, and age_days.*'''

# Push documentation updates to the server
client.update_model_version(
    name=model_name,
    version=version,
    description=description
)

# Apply metadata tags for audit trails
client.set_model_version_tag(model_name, version, 'validation_status', 'passed')
client.set_model_version_tag(model_name, version, 'team', 'datascience')
client.set_model_version_tag(model_name, version, 'framework', 'xgboost')

print('✓ Documentation and metadata tags successfully added.')

# Transition the model version state to "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage='Staging',
    archive_existing_versions=True
)

print(f'✓ Model version {version} successfully moved to [Staging] stage!')

✓ Documentation and metadata tags successfully added.
✓ Model version 1 successfully moved to [Staging] stage!


C:\Users\wayna\AppData\Local\Temp\ipykernel_18528\397503190.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [4]:
# Load the model directly using its alias stage string
staging_model = mlflow.pyfunc.load_model(
    model_uri=f'models:/{model_name}/Staging'
)
print('✓ Staging model downloaded and loaded successfully!')
print("-" * 60)

# Create high-risk hardware environment simulation parameters
test_data = pd.DataFrame({
    'temperature': [95.0],  # Critical Limit Exceeded
    'vibration': [0.9],    # Severe structural vibration
    'pressure': [135.0],   # Overpressurized
    'rpm': [1500.0],
    'age_days': [320]      # Way past maintenance window
})

# Execute inference run
prediction = staging_model.predict(test_data)

print('Test Prediction (Staging Model Evaluation):')
print(f'Input Context: High Temp, High Vibration, Aging Machinery')
print(f'Prediction Output: {"⚠️ FAILURE LIKELY" if prediction[0] == 1 else "✅ NORMAL OPERATION"}')

✓ Staging model downloaded and loaded successfully!
------------------------------------------------------------
Test Prediction (Staging Model Evaluation):
Input Context: High Temp, High Vibration, Aging Machinery
Prediction Output: ⚠️ FAILURE LIKELY


C:\Users\wayna\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [5]:
# Promote our verified version to Production stage
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage='Production',
    archive_existing_versions=True
)

print(f'🚀 Model version {version} has officially gone live in PRODUCTION!')

# Log deployment timestamp metadata
client.set_model_version_tag(
    model_name, version,
    'deployment_date',
    datetime.now().strftime('%Y-%m-%d')
)
print("-" * 60)

def predict_equipment_failure(temperature, vibration, pressure, rpm, age_days):
    """
    Production inference function.
    Dynamically routes inputs to whatever version is active in Production.
    """
    # Point directly to the Production alias URI
    prod_model = mlflow.pyfunc.load_model(
        model_uri=f'models:/PredictiveMaintenance/Production'
    )
    
    # Structure features identically to training schema
    input_data = pd.DataFrame([{
        'temperature': temperature,
        'vibration': vibration,
        'pressure': pressure,
        'rpm': rpm,
        'age_days': age_days
    }])
    
    # Predict failure binary output
    prediction = prod_model.predict(input_data)[0]
    
    return {
        'will_fail': bool(prediction),
        'recommendation': '🚨 EMERGENCY: Schedule immediate maintenance!' if prediction else '💚 System Stable: Normal operation.'
    }

# Test across distinct field runtime telemetry environments
scenarios = [
    {'name': 'Safe/Normal', 'temp': 70, 'vib': 0.4, 'press': 95, 'rpm': 1500, 'age': 100},
    {'name': 'Critical Risk', 'temp': 95, 'vib': 0.9, 'press': 135, 'rpm': 1500, 'age': 320},
    {'name': 'Warning/Medium', 'temp': 85, 'vib': 0.6, 'press': 110, 'rpm': 1500, 'age': 200}
]

print('Live Production Inference Test Routing:')
for s in scenarios:
    result = predict_equipment_failure(
        s['temp'], s['vib'], s['press'], s['rpm'], s['age']
    )
    print(f"Scenario: {s['name']:14} → Recommendation: {result['recommendation']}")

C:\Users\wayna\AppData\Local\Temp\ipykernel_18528\2522294119.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


🚀 Model version 1 has officially gone live in PRODUCTION!
------------------------------------------------------------
Live Production Inference Test Routing:


Scenario: Safe/Normal    → Recommendation: 🚨 EMERGENCY: Schedule immediate maintenance!


C:\Users\wayna\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Scenario: Critical Risk  → Recommendation: 🚨 EMERGENCY: Schedule immediate maintenance!


C:\Users\wayna\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Scenario: Warning/Medium → Recommendation: 🚨 EMERGENCY: Schedule immediate maintenance!


C:\Users\wayna\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [6]:
# Grab our runner-up model run metadata (Index 1)
second_run = runs[1]
model_uri_v2 = f'runs:/{second_run.info.run_id}/model'

# Register this runner up to simulate version updates
v2 = mlflow.register_model(model_uri_v2, model_name)
print(f'🆕 A new engineering version was submitted to registry as: Version {v2.version}')
print("-" * 60)

print('🚨 EMERGENCY ROLLBACK TRIGGERED!')
print('Routing global "Production" alias keyword backward to Version 1...')

# Shift our older stable Version 1 model straight back into the live Production slot
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage='Production',
    archive_existing_versions=True
)

print('\n[SUCCESS] Rollback procedure fully executed!')
print('✓ Production environments are safely using Version 1 again.')

Registered model 'PredictiveMaintenance' already exists. Creating a new version of this model...
2026/06/07 10:20:08 WARNING mlflow.tracking._model_registry.fluent: Run with id ea7990b7c63942aabc484bc14924d1ee has no artifacts at artifact path 'model', registering model based on models:/m-bece1a7559ac4c92a2879cd91163bf47 instead
2026/06/07 10:20:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PredictiveMaintenance, version 2
Created version '2' of model 'PredictiveMaintenance'.
C:\Users\wayna\AppData\Local\Temp\ipykernel_18528\2309453949.py:14: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_

🆕 A new engineering version was submitted to registry as: Version 2
------------------------------------------------------------
🚨 EMERGENCY ROLLBACK TRIGGERED!
Routing global "Production" alias keyword backward to Version 1...

[SUCCESS] Rollback procedure fully executed!
✓ Production environments are safely using Version 1 again.
